# Machine learning from start to finish
Earlier workshops focussed on building a model and quantifying performance.
This, of course, is not a goal in it self. Models are a means to an end; they only become useful when you start using them.
In this workshop, you will go through all major steps involved in setting up a model that is ready to be used.

## Learning objectives
- Get acquinted with revision control
- Learn to build images and how to use containers
- Train a model on the cloud
- Serve your predictions on the cloud

## Goal: setting up a digit classifier
The running example in this workshop will be to build and deploy an algorithm that reads black-and-white images of 8x8 pixels and detects which digit (numbered from 0-9) is displayed.


## Organization
During this workshop, you will work together in pairs (`student A` and `student B`). Decide who of you is `student A` and who is `student B`. For part of the instructions, you will have to work in parallel and complete different instructions. You will collaborate on this project using revision control system `git`, so that you can work on a single code base independently. Unless specified explicitly, you will both need to complete the steps.

## Google Cloud Platform credits
You should have received instructions with a Student Coupon Retrieval Link for $50 dollars in Google Cloud credits that you can redeem with your @student.rug.nl e-mail address. Make sure you have redeemed your cloud credits before you proceed.

**Important**: To prevent unnecessary costs, it is important to clean up your resources after you complete the workshop. This not only includes deployed end points (which consumes compute resources), but also other resources such as the cloud storage buckets and the container registry.

## Python imports

In [ ]:
import json
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score

## Setting up a repository
Before you continue, make sure:
- `student A`: Create a [new Github repository](https://github.com/new).
- `student A`: Invite `student B` by clicking Settings -> [Collaborators](https://github.com/hylkedonker/machine-learning-on-gcp/settings/access) -> Add people -> And click _Add people_ under Manage access.
- `student B`: Accept the invitation.

## `Both`: Configure Google Cloud Platform settings
Cloud computing and storage are based on a pay-as-you-go payment model.
To prevent unnecessary costs, you need to clean up your resources after you're done with them.

To make it easy to remove all resources together, I advise you to create a new project by:
1. Navigating to https://console.cloud.google.com/ and logging in with your credentials that you used to redeem your Google Cloud coupon.
2. Opening the project picker on the top left corner (Ctrl o).
3. Clicking `New project` on the top right corner of the popup menu.

After you've created a new project, make sure it is selected.
### Enter the cloud shell
Most of the work will be done in the cloud shell of GCP.
Open it by tapping `g s` or clicking on the icon in the top right corner with the _Activate cloud shell_ tool tip. In the terminal, enter the following:

```shell
###########################################################
# BOTH: Specify your new Google Cloud Platform project ID #
###########################################################
PROJECT_ID=...
```

Set our project ID and the Groningen Eemshaven data centre as the default.  
When prompted to retry, choose `y`.

```shell
gcloud config set project ${PROJECT_ID}
gcloud config set compute/zone europe-west4
```

## `Both`: Configure git

Next, configure git in the Google Cloud shell by providing some details to Git about who you are. Edit the placeholders and then run the following commands in the cloud shell:

```shell
# BOTH: Run in the Google cloud shell.
git config --global user.email "you@student.rug.nl"
git config --global user.name "Your Name"
```

In order to connect to the Github repository:
1. go to https://github.com/settings/tokens and generate a new `personal access token (classic)`.
2. Fill the Notes field and Select `repo` under scopes and click `Generate Token`.
3. Fill your token (in the green box) in the variable below:

In [ ]:
############################################################
# Fill out the variable below, and run in the cloud shell. #
############################################################
TOKEN=...

And run it in the cloud shell.

Next, given the Github `<username>` of `Student A` and your new Github token, clone your Git repository `<repo-name>`.  
**NB 1**: The text `${TOKEN}` is not a placeholder; keep it as is. The dollar sign with curly braces indicates: subtitutes the value of the variable `TOKEN` in the terminal command.  
**NB 2**: Replace the placeholders, which starts from `<` and ends with the `>` symbol.

```shell
git clone https://${TOKEN}@github.com/<username>/<repo-name>.git
```

To work with git, you need move in to the directory you just cloned from Github. Use the command `cd` (change directory) in the cloud terminal to move in.

```shell
cd <repo-name>
```

## `Student B`: Writing a readme and committing it to git.
`Student B` is going to add a README.md file to your repository. This document is typically used to explain what this repository is about, and will serve as the landing page when you navigate your browser to your github repo.
Click `Open editor`, and add a new file named `README.md`, and populate it with some information about your repository.

**Important**: Make sure to save the file in the directory that you cloned from Github.

Next, `student B` is going to commit this file to the repository.

To do this, `student B` needs to complete the following three steps (consult the [Pro Git book](https://github.com/progit/progit2/releases/download/2.1.448/progit.pdf) if needed) in the Google cloud shell:  
**NB**: To return from the editor back to the command line, click `Open Terminal` in the Google cloud shell.  
1. `add` the document to staging area.


```shell
##############################################
# Student B: Complete the following command: #
##############################################
git ...
```


2. `commit` your change to your local git repository. Use the `--message=` flag to provide a commit message.  
**NB**: If you don't specify a commit message, git will automatically open the _nano_ text editor, which allows you to provide a commit message.

```shell
##############################################
# Student B: Complete the following command: #
##############################################
git ...
```


3. `push` your local changes to the remote Github server.


```shell
##############################################
# Student B: Complete the following command: #
##############################################
git ...
```

`student A` can now `pull` in the changes `student B` made to his/her local repository.

```shell
##############################################
# Student A: Complete the following command: #
##############################################
git ...
```

If successfull, both of you should now have a `README.md` file in your directory. Verify this by running the following command:

```shell
# Both: Run the following command in the Google Cloud shell to verify you have the file.
ls -l README.md
```

## Building a prototype
Before training a model on the cloud, start by building a small prototype in this colab notebook. This will help you to quickly test ideas and get things working, before having to worry about all the overhead of cloud computing.

### Downloading the data.
Our dataset is a set of black-and-white digits, numbered 0-9. The images are 8x8 pixels, of 1 byte (= 4 bits) each.

Let us first download the data. Run the following cell in Google Colab:

In [ ]:
# BOTH: Run in Google Colab (NOT in the cloud shell).
# Download the files to Google Colab.
!gdown 1v81cbkumeKAbeMzr7w4Zz6J7WBYqO4oM --output digits_train.csv
!gdown 1iJmdvIbY_nbVWbT8aG8u9-_PtmEEcCv2 --output digits_test.csv

# Load the training set from a comma-separated file using pandas.
Xy_train = pd.read_csv('digits_train.csv')
y_train = Xy_train.pop('label')
X_train = Xy_train

And take a quick look at what the images look like.

In [ ]:
# BOTH: Run in Google Colab.
n_rows = 2
n_cols = 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 5))
for i in range(n_rows):
    for j in range(n_cols):
        idx = i*n_cols + j
        # Convert dataframe to NumPy matrix of floating points.
        x_train_idx = X_train.iloc[idx].to_numpy()
        axes[i, j].imshow(x_train_idx.reshape(8, 8), cmap='gray_r')
        axes[i, j].set_title(f'y = {y_train.loc[idx]}')
        axes[i, j].axis('off')

Next, train (or, `fit`) a simple `LogisticRegressionCV` model on the digits to predicts the digit number corresponding to the pixel intensities of the image.  
**Extra instruction**: Increase the number of steps, `max_iter` to `1_000`.

In [ ]:
#####################################################
# Both: Complete the following code in Google Colab #
#####################################################
classifier = ...

Now that the model is trained, let's see how well it can actually predict the digits correctly on an unseen test set of $m$ images. For now, we measure performance in terms of accuracy $a$:
$$
a = m^{-1} \sum_{i=1}^m \delta(y^{(i)}, \hat{y}^{(i)})
$$
which simply counts the proportion of labels predicted by the model ${\hat{y}}^{(i)}$ that match the ground truth $y^{(i)}$.

In [ ]:
# BOTH: Run the following code in Google Colab.
# Load test set from disk.
Xy_test = pd.read_csv('digits_test.csv')
y_test = Xy_test.pop('label').to_numpy()
X_test = Xy_test.to_numpy()

# Compute accuracy on test set.
classifier.score(X_test, y_test)

## Setting up cloud infrastructure
Next, we are going to save both the dataset, and later also the model itself, in a cloud storage bucket.

First, think of a unique identifier `<my-bucket>` to use as a name for your storage bucket. Then, issue the make bucket (`mb`) command in the cloud shell: `gsutil mb` to create it.

**Tip**: You can type `gsutil mb --help` to read help instructions.  
**Warning**: Your `<my-bucket>` name must globally unique across all Google Cloud projects. Since both of you will work in independent cloud projects, your bucket names must be different.  
**NB**: You can only use lower case alpha-numeric characters and dashes (no capital letters).

```shell
#############################################################
# Both: Complete the following code and run in cloud shell. #
#############################################################
gsutil mb -l europe-west4 gs://...
```

Install gdown and download the training data to your cloud shell.

```shell
################################################################
# Both: Complete the following code and run in the cloud shell #
################################################################
pip3 install gdown
export PATH=$PATH:$HOME/.local/bin
gdown 1v81cbkumeKAbeMzr7w4Zz6J7WBYqO4oM --output digits_train.csv
```

Next, use `gsutil cp` to copy the training set to your bucket. Save it under the folder `data/`.

```shell
################################################################
# Both: Complete the following code and run in the cloud shell #
################################################################
gsutil cp digits_train.csv gs://...
```


## Model training and deployment on the cloud
You will now collaborate by dividing tasks:
- `student B`: Writes the training script.
- `student A`: Will create an image that can be used to train the model.

You will do this by creating a pull request. That is, each of you will make a separate code branch that includes the changes you want to make to a shared code repository.
You then propose to include this change in the `main` (primary) branch, by reviewing each others work and then merging the branches.

## `Student B`: Writing the trainer task
**NB**: `Student A` can skip this section up to the point that he/she has to review `Student B`'s pull request.

You will write Python script called `trainer.py` that trains and saves a model checkpoint to Google Cloud Platform. When succesful, the trained model will show up in the [Vertex AI Model registry](https://console.cloud.google.com/vertex-ai/models) in the GCP console.

First, let's create a directory (`mkdir`) called _src/_ in your git repository, to store the source code.


```shell
# Student B: Create a directory to keep the Python files.
mkdir --parents src/
```

Two things:
- Pandas can read files directly from Google Cloud Storage, when you specify the URI with the _gs://_ prefix.

- Sci-kit learn models trained with Google Vertex AI are automatically detected when:
    1.  they are saved under under the name `"model.joblib"` and
    2. located in the directory indicated by environment variable `$AIP_MODEL_DIR`.

Let's use that. Click `Open Editor` in the cloud shell, complete the trainer script below and save the contents under `src/trainer.py`.

```python
import os
from google.cloud import storage
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegressionCV

##########################################
# Student B: Complete the following code #
##########################################
# Load the dataset from your Google Cloud Storage bucket.
Xy_train = pd.read_csv('gs://.../data/digits_train.csv')
# Instantiate classifier.
classifier = ...
##########################################

y_train = Xy_train.pop('label')
X_train = Xy_train
classifier.fit(X_train, y_train)

joblib.dump(classifier, 'model.joblib')

# Upload model artifact to Google Cloud Storage bucket.
model_directory = os.environ['AIP_MODEL_DIR']
storage_path = os.path.join(model_directory, 'model.joblib')
blob = storage.blob.Blob.from_string(storage_path, client=storage.Client())
blob.upload_from_filename('model.joblib')
```

Next, open a pull request to add your trainer Python script to the repository. Do this by completing the following steps (click `Open Terminal` to return to the cloud shell):
1. `switch` to a **new** branch called "feature/trainer" (use the option `--create`).

```shell
#####################################################################
# Student B: complete the following command and run in cloud shell. #
#####################################################################
git switch ...
```

2. Stage the file `trainer.py` and `commit`.  
**NB:** Don't forget to specify a commit message.


```shell
#####################################################################
# Student B: complete the following command and run in cloud shell. #
#####################################################################
git ...
git ...
```

3. `push` the new branch to Github.

```shell
#####################################################################
# Student B: complete the following command and run in cloud shell. #
#####################################################################
git push ...
```

4. Open pull request by visiting the webpage (replace the place holders) that is printed to the terminal:

https://github.com/<username\>/\<repo-name\>/pull/new/feature/trainer.

<!-- Click on the green button `Compare & pull request`.  -->

Write a short description, and under **_Reviewers_** (right hand side), select `Student A`. Then click the green `Create pull request` button.



5. Ask `Student A` to review your pull request, and then to merge the pull request in to the `main` branch.
6. Navigate back to your Google cloud shell. Then `switch` back to the main branch, by running:


```shell
git switch main
```

7. After `Student A` integrated the pull request in your `main` branch, synchronize your local repository with the remote Github repository.

```shell
# Get updates from the central repository.
git pull
```

## `Student A`: Reviewing the changes on Github.
**NB**: Continue to the next section if `Student B` did not yet complete the pull request. Return to this section when requested for review.

It is now `student A`'s turn to review the proposed changes. Click the green `Add your review` button to start reviewing `student A`'s work.
Here, you can select lines to leave comments. An overall report can be left by clicking the green `Review changes` button. When satisfied, select `Approve` and click `Submit review`. Finally, click `Merge pull request` to integrate the changes into the main branch.


## `Student A`: Building the image
You are going to build an image that can be used to run the `trainer.py` script that `student B` is building to train the model on the cloud. To do this, you are going to create a file named _Dockerfile_ that specifies how to build your image. That is, the Dockerfile contains all the dependencies that are needed to run the training procedure of the algorithm. In our case, our algorithm depends on sci-kit learn and pandas.


**Task**: Complete the following Dockerfile by:
  1. installing `scikit-learn` and `pandas[gcp]` using pip and
  2. Copying trainer.py to the directory `/app/` in the image.  

**Important**: Make sure to specify the version that you are using. Use versions `1.5.2` and `2.3.3` for scikit-learn and pandas, respectively.  

In the cloud shell of Google Cloud Platform, click `Open Editor`. Complete the file below and save the contents under the name `Dockerfile` in the parent directory of your local git repository:

```dockerfile
# Student A: Complete the following Dockerfile
FROM ubuntu:25.10
# Set environment variable to prevent user input by switching to noninteractive mode.
ARG DEBIAN_FRONTEND=noninteractive

# Install the pip command.
RUN apt update \
  && apt install --yes --no-install-recommends \
    python3-pip=25.1.1+dfsg-1ubuntu2 \
    curl=8.14.1-2ubuntu1

##########################################################################
# TODO: Install pandas[gcp] and sci-kit learn Python package using `pip`.#
#       And copy the contents of the src directory in /app/
##########################################################################
RUN python3 -m pip install \
  --no-cache-dir \
  --break-system-packages \
  google-cloud-storage==3.4.1 \
  ...

COPY ...
##########################################################################

WORKDIR /app/

ENTRYPOINT ["python3", "/app/trainer.py"]
```

It is good practice to test if everything is working as expected. Try building it by running the following command in the cloud shell:

```shell
docker build \
  --file Dockerfile \
  .
```

**NB**: The file `trainer.py` is not yet in the repository until `Student B`'s pull request is merged into `main` and (ii) you `git pull`ed the new file into your local copy. So it is not yet possible to run the container until then.

Next, open a pull request to add your Dockerfile to the repository. Do this by completing the following steps:
1. `switch` to a **new** branch called "feature/docker" (use the `--create` option).


```shell
########################################################################
# Student A: complete the following command and run in the cloud shell #
########################################################################
git switch ...
```

2. Stage the new `Dockerfile` and `commit`.


```shell
########################################################################
# Student A: complete the following command and run in the cloud shell #
########################################################################
git ...
git ...
```


3. `push` the new branch to Github.

```shell
########################################################################
# Student A: complete the following command and run in the cloud shell #
########################################################################
git push ...
```

4. Open pull request by visiting the webpage (replace the place holders):

https://github.com/<username\>/\<repo-name\>/pull/new/feature/docker.

Click on the green button `Compare & pull request`. Write a short description, and under **_Reviewers_**, select `Student B`.

5. Go back to the main branch. Run:


```shell
git switch main
```

## `Student B`: Reviewing the changes.
It is now `student B`'s turn to review the proposed changes. Click the green `Add your review` button to start reviewing `student A`'s work.
Here, you can select lines to leave comments. An overall report can be left by clicking the green `Review changes` button. When satisfied, select `Approve` and click `Submit review`. Finally, click `Merge pull request` to integrate the changes into the main branch.

## `Both`: updating your local snapshot
The changes have now been made in the remote repository. However, both of your local repositories are not yet updated. `Pull` in the changes from Github by running:

```shell
############################################################################
# Both (A + B): Complete the following command and run in the cloud shell: #
############################################################################
git ...
```

## `Both`: Publishing the image
Now that we have both the trainer script and the Dockerfile to create the image, let's build the image using the command:


```shell
docker build \
  --file Dockerfile \
  --tag europe-west4-docker.pkg.dev/${PROJECT_ID}/docker/digit-trainer \
  .
```

We have tagged this image in anticipation of where the image is going to be stored. Namely, we plan to save it in the google cloud container registry named `docker` which is located in the computing centre named `europe-west4`.
Let's create this repository by executing the following command in the cloud shell:

```shell
gcloud artifacts repositories create \
  docker \
  --repository-format=docker \
  --location="europe-west4" \
  --description="Docker repository"
```

When prompted to activate an API, choose yes.

Expose the GCP artifact registry to Docker.
```shell
# Tell Docker that we plan to save images on europe-west4-docker.pkg.dev.
gcloud auth configure-docker europe-west4-docker.pkg.dev
```

Save image to container registry.
```shell
# Save the image that we have build on Google Cloud Platform, so that Vertex AI
# (below) has access.
docker push europe-west4-docker.pkg.dev/${PROJECT_ID}/docker/digit-trainer
```

## `Both`: Model training on GCP
Next, click on the hamburger menu ≡ icon (top left corner in Google Cloud), select `Vertex AI` > `Training` (under Model development). Enable the Vertex AI API when prompted to do so.

Next, click on `+ Train new model`. A new menu will pop up, with six steps (numbered from 1. Training Method to 6. Inference container).

Go through the steps:
1. `Training method`: select `Custom training (advanced)`, and leave the rest of the options to their default values. Then click Continue.
2. `Model details`: Select `Train new model` and enter a name, such as `"digits classifier"`, and give it a brief Description.
3. `Training container`:
    - Select `Custom container` and click Browse in the field `Container image` and select the Docker image that is tagged `latest` from the cloud storage bucket that is used to store the images from your container registry.
    - For the model output directory, browse to `<my-bucket>`, which you created in the previous step, and click `Create new directory` to make a new folder called _artifacts_ and click `Select`.
4. `Hyperparameters`: Ignore this for now and click `Continue`.
5. `Compute and pricing`: Select region `europe-west4 (Netherlands)` which is the computing centre in Eemshaven, Groningen. Under _Compute settings_, select `Deploy to new worker pool`. Since our model is trivial to train, we can work with cheapest `Machine type` available, which is `Standard>n1-standard-4`. Under _Availability policies_, change the VM provisioning model to Spot.
6. `Inference container`: Click pre-built container. Under `Model framework` select "scikit-learn" and the `Model framework version` to 1.5. The `Model directory` should have been correctly configured automatically.

You can now select blue button that says `Start training`.

It can take about five minutes or so to schedule and run the training pipeline.
Congrats, you've now succesfully trained your model on the cloud! 🎉



## `Both`: Deploying your model
Next, we are going to make your model available for online use (model serving). There are two elements to this:
- We need to set up an online server that hosts the model and produces predictions.
- And we need a way to communicate our input (the images) to the server and communicate back the response (the model prediction). This shared contract, how to communicate, is called an Application Programming Interface (**API**).

But first, let's create an endpoint. An endpoint is the point of contact between the consumer (user) of the model and the server that hosts the model.

Click on `Model Registry` in the Vertex AI side panel, and select your trained model from the list.

**N.B.**: If there is no model in the list, then your model might still be training.

Next, click the `Deploy & test` tab, and click the blue `Deploy to endpoint` button.
1. `Define your endpoint`: give your endpoint a name and click _Continue_.
2. `Model settings`: For the `Machine type`, select `Standard > n1-standard-2` and click Continue .
3. Disable model monitoring for now and click `Deploy`.

**N.B.** It may take a while for your model is done deploying the model and reach the status _Active_.

## `Both`: Using your model

If you completed the last steps succesfully, you have now deployed a REST (Representational State Transfer) API endpoint and is ready for use.  This means that you can start sending images to your server and get back the result.

Let's do that on a fresh set of data.

Run the following code in Google Colab.

In [ ]:
# Download the CSV to Google Colab.
!gdown 1YDHaXmgPascbgauVRx4UlTSOa8IiGqbZ --output digits_fresh.csv

In [ ]:
# Load fresh data from disk.
Xy_fresh = pd.read_csv('digits_fresh.csv')
y_fresh = Xy_fresh.pop('label').to_numpy()
X_fresh = Xy_fresh.to_numpy()

There are several ways to inferface with the endpoint. But the simplest way, and the one that we will be using, is by sending JSON data.
JSON is short for JavaScript Object Notation, and is essentially a nested dictionary.
The format that your endpoint requires is the following:
```json
{
    'instances': [image1, image2, ...]
}
```
where `image1` is a `list` of decimal point numbers.

But first, make sure you are authenticated from Google Colab. Run this from colab, and follow the instructions (which involves coping the authentication code in the field that appears below):

In [ ]:
!gcloud auth application-default login

Next, on the `Endpoints` page in the Google Cloud console, click `Sample request` under the column API in your list of Endpoints. A new side panel should appear. Note the value of `ENDPOINT_ID` in code box 4. under the Tab `REST` and fill out the value below, and run the cell in Google Colab:

In [ ]:
################################################
# Both (A + B): Complete the following command:#
################################################
ENDPOINT_ID="..."


To communicate with the server, turn the images into a JSON format that the endpoint expects.

Run the following code in Google Colab:

In [ ]:
# Convert images to list, and turn into JSON.
images_as_lists = [image.tolist() for image in X_fresh]
request_payload = {'instances': images_as_lists}
INPUT_PAYLOAD = json.dumps(request_payload)

I invite you to take a look to see what `INPUT_PAYLOAD` looks like.

Next, we are going to send the payload of images to our endpoint using a HTTP POST request. We do that using the command line command `curl`.
The following cell runs the command and prints out the response given by the endpoint.

In [ ]:
! curl \
-X POST \
-H "Authorization: Bearer $(gcloud auth print-access-token)" \
-H "Content-Type: application/json" \
"https://europe-west4-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/locations/europe-west4/endpoints/{ENDPOINT_ID}:predict" \
--data "{INPUT_PAYLOAD}"

Put the predicted values that was returned by the endpoint in the variable `y_fresh_pred` and use the ground truth `y_fresh` to compute the accuracy using the `accuracy_score` function.

**Question**: What do you notice on the model performance?

**Answer**:

In [ ]:
################################################
# Both (A + B): Complete the following command:#
################################################
y_fresh_pred = ...
accuracy_score(y_fresh, y_fresh_pred)



To investigate further, plot some of the images and their predictions.

**Question**: Can you tell why the model performance is what it is?  
**Answer:**

In [ ]:
n_rows = 2
n_cols = 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 5))
for i in range(n_rows):
    for j in range(n_cols):
        idx = i*n_cols + j
        axes[i, j].imshow(X_fresh[idx].reshape(8, 8), cmap='gray_r')
        predicted_label = y_fresh_pred[idx]
        true_label = y_fresh[idx]
        axes[i, j].set_title(f'Predicted: {predicted_label}\nTrue: {true_label}')
        axes[i, j].axis('off')

plt.tight_layout()
plt.show()

## Clean up
To prevent unnecessary costs, it is important to clean up all your resources.
Visit [IAM & Admin > Settings](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects#:~:text=In%20the%20Google%20Cloud%20console%2C%20go,the%20IAM%20%26%20Admin%20Settings%20page.&text=IAM%20%26%20Admin%20Settings-,The%20remaining%20steps%20appear%20in%20the%20Google%20Cloud%20console.,and%20then%20click%20Shut%20down.) and click `Shutdown` to remove your project, so that you won't be billed for the project.